# Test Module 1 Segmentation on Kaggle

Notebook nay chay YOLOv11-Seg tren mot anh, xuat dung Module 1 JSON va hien thi overlay, mask, crop cua tung vien.

Truoc khi chay, attach hai Kaggle Dataset: mot Dataset chua checkpoint `.pt` va mot Dataset chua anh test. Notebook tu clone source code tu GitHub vao `/kaggle/working`.

In [ ]:
from pathlib import Path

GIT_REPO_URL = 'https://github.com/GOx9-P/Multiple-Pill-Recognition-And-Interaction-Safety.git'
REPO_DIR = Path('/kaggle/working/Multiple-Pill-Recognition-And-Interaction-Safety')

# Sua hai path nay theo ten Kaggle Dataset cua ban.
SEGMENTATION_WEIGHTS = Path(
    '/kaggle/input/pill-segmentation-model/yolov11m_seg_mediseg_full_finetune_v1.pt'
)
IMAGE_PATH = Path('/kaggle/input/pill-test-images/pills.jpg')

# Tat ca artifact inference se ghi vao working directory co quyen ghi.
OUTPUT_DIR = Path('/kaggle/working/pill_segmentation_outputs')

REQUEST_ID = 'req_kaggle_001'
SESSION_ID = 'kaggle_segmentation_test'
IMAGE_ID = IMAGE_PATH.stem

print('REPO_DIR:', REPO_DIR)
print('SEGMENTATION_WEIGHTS:', SEGMENTATION_WEIGHTS)
print('IMAGE_PATH:', IMAGE_PATH)


In [ ]:
# Kaggle Notebook Settings phai bat Internet de clone GitHub.
import subprocess

if not REPO_DIR.is_dir():
    subprocess.run(
        ['git', 'clone', '--depth', '1', GIT_REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print(f'Repository da ton tai, bo qua clone: {REPO_DIR}')

print('Repository ready:', REPO_DIR)


In [ ]:
# Chay cell nay mot lan neu Kaggle image chua co Ultralytics.
# Can bat Internet trong Notebook Settings neu pip can tai package.
%pip install -q ultralytics==8.3.253 pyyaml==6.0.2

import sys
import torch

print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Kiem tra tai nguyen truoc khi import code project.
if not REPO_DIR.is_dir():
    raise FileNotFoundError(
        f'Repo khong ton tai: {REPO_DIR}. Chay lai cell clone va kiem tra Internet setting.'
    )
if not SEGMENTATION_WEIGHTS.is_file():
    raise FileNotFoundError(f'Khong tim thay checkpoint: {SEGMENTATION_WEIGHTS}')
if not IMAGE_PATH.is_file():
    raise FileNotFoundError(f'Khong tim thay anh test: {IMAGE_PATH}')

SRC_DIR = REPO_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from pill_safety.cv.segmentation import SegmentationConfig, SegmentationPredictor
from pill_safety.schemas import SegmentationInferenceRequest

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Project source:', SRC_DIR)
print('Output directory:', OUTPUT_DIR)


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

with Image.open(IMAGE_PATH) as source:
    image = source.convert('RGB')

plt.figure(figsize=(10, 7))
plt.imshow(image)
plt.title(f'Input image: {IMAGE_PATH.name}')
plt.axis('off')
plt.show()


In [ ]:
# Tao request truc tiep tu IMAGE_PATH; khong can tao request.json.
request = SegmentationInferenceRequest(
    request_id=REQUEST_ID,
    session_id=SESSION_ID,
    image_id=IMAGE_ID,
    image_path=str(IMAGE_PATH),
)

config = SegmentationConfig.from_yaml(
    REPO_DIR / 'configs' / 'inference' / 'segmentation.yaml'
).with_weights_path(SEGMENTATION_WEIGHTS).with_output_dir(OUTPUT_DIR)

# Config YAML da dung device=auto: Ultralytics se dung GPU neu Kaggle cap GPU.
predictor = SegmentationPredictor(config=config)
artifacts = predictor.predict_with_artifacts(request)
result = artifacts.output.model_dump(mode='json')

print('Detected instances:', len(result['instances']))
print('Schema JSON:', artifacts.schema_json_path)
print('Overlay:', artifacts.overlay_path)


In [ ]:
# Hien thi overlay va thong tin tong quat.
if artifacts.overlay_path is not None and artifacts.overlay_path.exists():
    with Image.open(artifacts.overlay_path) as source:
        overlay = source.convert('RGB')
    plt.figure(figsize=(12, 8))
    plt.imshow(overlay)
    plt.title('Segmentation overlay')
    plt.axis('off')
    plt.show()

display({
    'image_quality': result['image_quality'],
    'instance_count': len(result['instances']),
})


In [ ]:
# Hien thi mask/crop cua tung vien va evidence can dung cho Module 2/3.
instances = result['instances']
if not instances:
    print('Khong phat hien vien nao. Kiem tra checkpoint, confidence_threshold va anh input.')
else:
    figure, axes = plt.subplots(len(instances), 2, figsize=(10, 5 * len(instances)))
    if len(instances) == 1:
        axes = [axes]

    for row, instance in zip(axes, instances):
        with Image.open(instance['mask_path']) as source:
            mask = source.convert('L')
        with Image.open(instance['crop_path']) as source:
            crop = source.convert('RGB')

        row[0].imshow(mask, cmap='gray')
        row[0].set_title(f"{instance['instance_id']} mask")
        row[0].axis('off')
        row[1].imshow(crop)
        row[1].set_title(
            f"{instance['instance_id']} crop | conf={instance['segmentation']['confidence']:.3f}"
        )
        row[1].axis('off')

    plt.tight_layout()
    plt.show()

    for instance in instances:
        print('\n', instance['instance_id'])
        display({
            'bbox_xyxy': instance['bbox_xyxy'],
            'segmentation': instance['segmentation'],
            'quality_flags': instance['quality_flags'],
            'mask_path': instance['mask_path'],
            'crop_path': instance['crop_path'],
        })


In [ ]:
# Module 1 JSON nay la input de tao request cho Attribute va OCR o buoc sau.
import json

print(json.dumps(result, indent=2, ensure_ascii=False))
print('\nSaved JSON:', artifacts.schema_json_path)
